# 01 - Parsing: How PDFs really get processed

**Aligns with**: S4 Sec. 3.2 and 3.3 | **Estimated time**: 30 minutes | **Estimated cost**: ~$0.10

## What 00_quickstart left unanswered

In `00_quickstart` we used `PyMuPDFLoader` to load the Wells Fargo PDF in one
line. But **do the chunks you retrieve actually contain complete information?**

Let's run an experiment to see what the quickstart RAG **cannot** answer.


## Step 1: See what the quickstart missed

Ask a question whose answer **only exists in a table**: what is ROTCE?


In [1]:
# Path setup: notebook is in notebooks/, src/ lives at ../src/
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Reproduce the quickstart RAG (default PyMuPDFLoader behavior)
from langchain_community.document_loaders import PyMuPDFLoader

raw_docs = PyMuPDFLoader(str(ROOT / "data/uploads/wells_fargo.pdf")).load()

# What does PyMuPDFLoader give us for page 1?
print("=" * 60)
print("PyMuPDFLoader page 1 output (first 1500 chars)")
print("=" * 60)
print(raw_docs[0].page_content[:1500])

PyMuPDFLoader page 1 output (first 1500 chars)
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
  
News Release | January 14, 2026 
Wells Fargo Reports Fourth Quarter 2025 Net Income of 
$5.4 billion, or $1.62 per Diluted Share 
Net income, excluding a notable item, of $5.8 billion, or $1.76 per diluted 
share1 
Company-wide Financial Summary 
Selected Income Statement Data 
($ in millions except per share amounts) 
Quarter ended 
Dec 31, 
Dec 31, 
2025 
2024 
Total revenue 
$ 21,292 
20,378 
Noninterest expense 
13,726 
13,900 
Provision for credit losses2 
1,040 
1,095 
Net income 
5,361 
5,079 
Diluted earnings per common share 
1.62 
1.43 
Selected Balance Sheet Data 
($ in billions) 
Average loans 
Average deposits 
CET13 
$ 
955.8 
1,377.7 
10.6% 
906.4 
1,353.8 
11.1 
Performance Metrics 
ROE4 
12.3% 
11.7 
ROTCE5 
14.5 
13.9 
Operating Segments and Other Highlights 
Quarter 
Dec 31, 2025 
ended 
% Change from 
($ in billions) 
Dec 31, 
2025 
Sep 30,
Dec 31, 
2025 
2024 
Average loan

Notice anything? **The tables are squashed into a mess.** The Selected Income
Statement Data table - every number, every header - is jammed together with
surrounding text.

Worse: **images are gone entirely**. For something like a Tesla deck with factory
photos, PyMuPDFLoader treats those pages as if they only contain a few caption lines.

**This is why we need to rewrite the parsing layer.**


## Step 2: Our industrial-grade ingestion pipeline

Introduce `IngestionPipeline` - it does three things LangChain's defaults skip:

1. **Structural parsing**: detects heading / paragraph / table / image and
   **preserves hierarchy**
2. **VLM caption**: tables and images get a natural-language summary from a
   vision-capable model, used for embedding
3. **Three-layer cache**: document / VLM / embedding cache, all content-addressed,
   so **a student rerunning the lab pays $0**

The architecture:

```
[ PDF bytes ]
      |
   Loader (PyMuPDF)         <- Stage 1: extract raw text + tables + images
      |
   Parser (Structural)      <- Stage 2: detect block types, build heading_path
      |
   Captioner (VLM)          <- Stage 3: tables/images -> semantic_content
      |
   Document {blocks: [...]}
```

Each stage is an **implementation of an abstract base class**
(`BaseLoader` / `BaseParser` / `BaseCaptioner`), independently swappable and
mockable in tests.


In [3]:
from src.pipelines.ingestion import IngestionPipeline

# Default config: PyMuPDFLoader + PDFStructuralParser + GPT5MiniCaptioner + 3-layer cache (on)
pipeline = IngestionPipeline()

# One-line ingestion
report = pipeline.ingest(
    str(ROOT / "data/uploads/wells_fargo.pdf"),
    verbose=True,
)
print()
print(report.summary())

 cache HIT for wells_fargo.pdf -> eb705b11a230...

Document: Wells Fargo Reports Fourth Quarter 2025 Net Income of $5.4 billion, or $1.62 per Diluted Share (12 pages)
 Blocks: 418 text, 0 tables, 1 images
 Chunks: 0
 VLM cache: 0 hits / 0 misses
 Embed cache: 0 hits / 0 misses
Cost: $0.0000    time 0.0s


**Key observations**:

- `n_text_blocks` is much larger than `len(raw_docs)` - we split each page into
  multiple semantic blocks
- `n_table_blocks` > 0 means **tables were detected** as standalone blocks
  (note: WF press releases use positioned-text tables; PyMuPDF's table detector
  may report 0 here, which is itself a useful signal we'll work with in 02)
- Run this cell a second time: `parse_cache_hit=True`, 0 seconds, $0 spent


## Step 3: Inspect the Document structure

In [4]:
doc = report.document
print(f"Title: {doc.title}")
print(f"  {doc.n_pages} pages, {len(doc.blocks)} total blocks")
print(f"  source_hash: {doc.source_hash[:16]}...")
print()

# Counts by block type
from collections import Counter
type_counts = Counter(b.block_type for b in doc.blocks)
for t, n in type_counts.most_common():
    print(f"  {t:12s}: {n}")

Title: Wells Fargo Reports Fourth Quarter 2025 Net Income of $5.4 billion, or $1.62 per Diluted Share
  12 pages, 419 total blocks
  source_hash: eb705b11a23063f8...

  paragraph   : 386
  h2          : 32
  image       : 1


In [5]:
# Look at the first 5 blocks
print("First 5 blocks:")
for b in doc.blocks[:5]:
    print(f"  [{b.block_type}] page {b.page_number}, heading_path={b.heading_path}")
    print(f"      {b.display_text(max_chars=120)}")

First 5 blocks:
  [paragraph] page 1, heading_path=[]
      [paragraph] News Release | January 14, 2026 Wells Fargo Reports Fourth Quarter 2025 Net Income of $5.4 billion, or $1.62 per Diluted...
  [h2] page 1, heading_path=[]
      [h2] Company-wide Financial Summary
  [paragraph] page 1, heading_path=['Company-wide Financial Summary']
      [paragraph] Selected Income Statement Data ($ in millions except per share amounts)
  [h2] page 1, heading_path=[]
      [h2] Quarter ended
  [paragraph] page 1, heading_path=['Quarter ended']
      [paragraph] Dec 31, Dec 31,


## Step 4: What VLM captioning does - find a table block

Every `table` block has a `semantic_content` field, which is a natural-language
summary generated by a vision-capable LLM. Example:


In [6]:
# Find the first table block
table_blocks = doc.blocks_by_type("table")
print(f"Found {len(table_blocks)} table blocks\n")

if table_blocks:
    tb = table_blocks[0]
    print(f"Heading path: {' > '.join(tb.heading_path) or '(root)'}")
    print(f"Page: {tb.page_number}")
    print()
    print("=" * 60)
    print("Raw table (markdown format, stored in .text)")
    print("=" * 60)
    print(tb.text[:600])
    print()
    print("=" * 60)
    print("VLM-generated semantic_content (used for embedding)")
    print("=" * 60)
    print(tb.semantic_content if tb.semantic_content else "[no caption - probably no API key]")

Found 0 table blocks



**Why does this matter?**

Embedding a blob of `| col | col | col |` characters directly produces a
near-meaningless vector that won't match queries.

Embedding the NL summary "*Wells Fargo Q4 2025 Selected Income Statement Data
showing revenue, net income, EPS comparing Q4 2024 vs Q4 2025...*" makes vector
search instantly able to match queries like "what was Wells Fargo's revenue
change Q4".

**This is the caption-then-embed pattern from S4 Sec. 3.3.**


## Step 5: Cache in action - second run is $0

Test the cache:


In [7]:
import time

# Second run (already ingested above, should be a cache hit)
t0 = time.perf_counter()
report1 = pipeline.ingest(str(ROOT / "data/uploads/wells_fargo.pdf"))
t1 = time.perf_counter() - t0
print(f"Second run on same file:")
print(f"  Wall time: {t1*1000:.1f} ms")
print(f"  Cost: ${report1.total_cost_usd:.4f}")
print(f"  Cache hit: {report1.parse_cache_hit}")

Second run on same file:
  Wall time: 17.9 ms
  Cost: $0.0000
  Cache hit: True


## Step 6: page_range parameter - process specific pages only

Financial Statements typically appear in the last few pages of an earnings PDF.
If you only care about the numbers, you can ingest just those pages, saving
VLM cost and time.


In [8]:
# Process only the first 3 pages
report_partial = pipeline.ingest(
    str(ROOT / "data/uploads/wells_fargo.pdf"),
    max_pages=3,
)
print(f"First 3 pages: {report_partial.document.n_pages} pages, {len(report_partial.document.blocks)} blocks")

# Note: cache key includes max_pages, so this won't collide with the full-doc cache
report_full = pipeline.ingest(str(ROOT / "data/uploads/wells_fargo.pdf"))
print(f"Full file: {report_full.document.n_pages} pages, {len(report_full.document.blocks)} blocks")

First 3 pages: 3 pages, 171 blocks
Full file: 12 pages, 419 blocks


## Step 7: Drop in your own PDF

The whole pipeline is designed for **"any PDF in, structured Document out"**.
Switching documents is just a path change:

```python
# For example, Tesla Q1 2026
report = pipeline.ingest("data/uploads/tesla.pdf")

# Or a Microsoft 10-K for an interview demo
report = pipeline.ingest("/path/to/msft_10k.pdf")
```

**This is what production ingestion looks like** - not a hard-coded file but a
service that ingests arbitrary PDFs. In `06_observability` we wrap this as a
FastAPI endpoint and turn it into a real service.


## Exercise 1.A

Ingest the Wells Fargo PDF with **different max_pages** values:

1. `max_pages=2` (first 2 pages only)
2. `max_pages=5` (first 5 pages)
3. Unbounded (all ~10 pages)

Verify:
- The 3 cache keys are **independent** (no cross-pollution)
- After running unbounded once, running with `max_pages=2` is **still a fresh run**
  (different cache key)
- After `clear_cache()`, all 3 versions need to re-run


In [ ]:
# TODO: your exercise code
# pipeline.clear_cache()  # clear
# report_a = pipeline.ingest(..., max_pages=2)
# report_b = pipeline.ingest(..., max_pages=5)
# report_c = pipeline.ingest(...)
# Then run the same three again and check parse_cache_hit is True


## Recap of the core concepts

| Concept | Where it lives |
|---|---|
| **S4 Sec. 3.2 Document -> Block hierarchy** | `Document.blocks: list[DocumentBlock]`, each Block has `block_type`, `heading_path`, `bbox` |
| **S4 Sec. 3.3 Caption-then-embed** | `Captioner` adds `semantic_content` to table/image blocks; `block.get_embed_text()` prefers caption |
| **Strategy pattern** | `BaseLoader` / `BaseParser` / `BaseCaptioner` abstract classes with multiple implementations |
| **Content-addressed cache** | Cache key = file SHA256 + parser config + captioner model. Change any input -> invalidate |
| **Cost transparency** | `report.total_cost_usd` and `report.cost_breakdown` make every dollar visible |

Next, `02_chunking.ipynb` answers the other quickstart question:
**Is chunk_size=500 optimal? How do you know?**
